# Feature Engineering

This notebook will generate team strength, form, and player-based features for model training and simulation.

In [ ]:
import pandas as pd

matches = pd.read_csv("../datasets/cleaned/cleaned_matches.csv")
players = pd.read_csv("../datasets/cleaned/cleaned_players.csv")
appearances = pd.read_csv("../datasets/cleaned/cleaned_appearances.csv")
def categorize_position(position):
    position = str(position)

    if any(pos in position for pos in ["CF", "ST", "LW", "RW"]):
        return "Attack"

    elif any(pos in position for pos in ["CM", "CDM", "CAM", "LM", "RM"]):
        return "Midfield"

    elif any(pos in position for pos in ["CB", "LB", "RB", "LWB", "RWB"]):
        return "Defense"

    elif "GK" in position:
        return "Goalkeeper"

    return "Other"
players["Position_Category"] = players["position"].apply(categorize_position)
players["player_strength"] = (
    players["market_value_in_eur"] / 1_000_000
)


In [ ]:
matches["date"] = pd.to_datetime(matches["date"])

In [ ]:
def calculate_recent_form(team, matches):
    """
    Calculate recent form statistics for a team.

    Features:
    - Wins / Draws / Losses
    - Goals scored / conceded
    - Clean sheets
    - Goal averages

    for:
    - last 5 overall matches
    - last 5 home matches
    - last 5 away matches
    """

    # ==================================================
    # OVERALL LAST 5 MATCHES
    # ==================================================

    overall_matches = matches[
        (matches["home_team"] == team) |
        (matches["away_team"] == team)
    ].sort_values("date", ascending=False).head(5)

    wins_overall = 0
    draws_overall = 0
    losses_overall = 0

    goals_scored_overall = 0
    goals_conceded_overall = 0

    clean_sheets_overall = 0

    for _, row in overall_matches.iterrows():

        if row["home_team"] == team:

            scored = row["home_score"]
            conceded = row["away_score"]

        else:

            scored = row["away_score"]
            conceded = row["home_score"]

        goals_scored_overall += scored
        goals_conceded_overall += conceded

        if conceded == 0:
            clean_sheets_overall += 1

        if scored > conceded:
            wins_overall += 1

        elif scored < conceded:
            losses_overall += 1

        else:
            draws_overall += 1

    total_overall = max(len(overall_matches), 1)

    avg_goals_scored_overall = (
        goals_scored_overall / total_overall
    )

    avg_goals_conceded_overall = (
        goals_conceded_overall / total_overall
    )

    # ==================================================
    # HOME LAST 5 MATCHES
    # ==================================================

    home_matches = matches[
        matches["home_team"] == team
    ].sort_values("date", ascending=False).head(5)

    wins_home = 0
    draws_home = 0
    losses_home = 0

    goals_scored_home = 0
    goals_conceded_home = 0

    clean_sheets_home = 0

    for _, row in home_matches.iterrows():

        scored = row["home_score"]
        conceded = row["away_score"]

        goals_scored_home += scored
        goals_conceded_home += conceded

        if conceded == 0:
            clean_sheets_home += 1

        if scored > conceded:
            wins_home += 1

        elif scored < conceded:
            losses_home += 1

        else:
            draws_home += 1

    total_home = max(len(home_matches), 1)

    avg_goals_scored_home = (
        goals_scored_home / total_home
    )

    avg_goals_conceded_home = (
        goals_conceded_home / total_home
    )

    # ==================================================
    # AWAY LAST 5 MATCHES
    # ==================================================

    away_matches = matches[
        matches["away_team"] == team
    ].sort_values("date", ascending=False).head(5)

    wins_away = 0
    draws_away = 0
    losses_away = 0

    goals_scored_away = 0
    goals_conceded_away = 0

    clean_sheets_away = 0

    for _, row in away_matches.iterrows():

        scored = row["away_score"]
        conceded = row["home_score"]

        goals_scored_away += scored
        goals_conceded_away += conceded

        if conceded == 0:
            clean_sheets_away += 1

        if scored > conceded:
            wins_away += 1

        elif scored < conceded:
            losses_away += 1

        else:
            draws_away += 1

    total_away = max(len(away_matches), 1)

    avg_goals_scored_away = (
        goals_scored_away / total_away
    )

    avg_goals_conceded_away = (
        goals_conceded_away / total_away
    )

    return {

        # OVERALL
        "wins_last5_overall": wins_overall,
        "draws_last5_overall": draws_overall,
        "losses_last5_overall": losses_overall,

        "goals_scored_last5_overall":
            goals_scored_overall,

        "goals_conceded_last5_overall":
            goals_conceded_overall,

        "clean_sheets_last5_overall":
            clean_sheets_overall,

        "avg_goals_scored_last5_overall":
            avg_goals_scored_overall,

        "avg_goals_conceded_last5_overall":
            avg_goals_conceded_overall,

        # HOME

        "wins_last5_home":
            wins_home,

        "draws_last5_home":
            draws_home,

        "losses_last5_home":
            losses_home,

        "goals_scored_last5_home":
            goals_scored_home,

        "goals_conceded_last5_home":
            goals_conceded_home,

        "clean_sheets_last5_home":
            clean_sheets_home,

        "avg_goals_scored_last5_home":
            avg_goals_scored_home,

        "avg_goals_conceded_last5_home":
            avg_goals_conceded_home,

        # AWAY

        "wins_last5_away":
            wins_away,

        "draws_last5_away":
            draws_away,

        "losses_last5_away":
            losses_away,

        "goals_scored_last5_away":
            goals_scored_away,

        "goals_conceded_last5_away":
            goals_conceded_away,

        "clean_sheets_last5_away":
            clean_sheets_away,

        "avg_goals_scored_last5_away":
            avg_goals_scored_away,

        "avg_goals_conceded_last5_away":
            avg_goals_conceded_away
    }
all_teams = sorted(
    set(matches["home_team"]).union(
        set(matches["away_team"])
    )
)
form_data = []

for team in all_teams:

    stats = calculate_recent_form(team, matches)

    stats["nationality"] = team

    form_data.append(stats)

form_features = pd.DataFrame(form_data)
team_features = team_features.merge(
    form_features,
    on="nationality",
    how="left"
)

In [ ]:
player_stats = (
    appearances
    .groupby("player_id")
    .agg({
        "minutes_played": "sum",
        "goals": "sum",
        "assists": "sum"
    })
    .reset_index()
)

In [ ]:
players_enriched = players.merge(
    player_stats,
    on="player_id",
    how="left"
)
players_enriched[
    ["minutes_played", "goals", "assists"]
] = players_enriched[
    ["minutes_played", "goals", "assists"]
].fillna(0)

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

players_enriched[
    [
        "market_value_norm",
        "minutes_norm",
        "goals_norm",
        "assists_norm"
    ]
] = scaler.fit_transform(
    players_enriched[
        [
            "market_value_in_eur",
            "minutes_played",
            "goals",
            "assists"
        ]
    ]
)
players_enriched["player_strength"] = (
      0.50 * players_enriched["market_value_norm"]
    + 0.25 * players_enriched["minutes_norm"]
    + 0.15 * players_enriched["goals_norm"]
    + 0.10 * players_enriched["assists_norm"]
)

In [ ]:
def top_n_average(df, n):

    if len(df) == 0:
        return 0

    return (
        df.nlargest(
            n,
            "player_strength"
        )["player_strength"]
        .mean()
    )

In [ ]:
attack_players = players_enriched[
    players_enriched["position"] == "Attack"
]

midfield_players = players_enriched[
    players_enriched["position"] == "Midfield"
]

defense_players = players_enriched[
    players_enriched["position"] == "Defender"
]

gk_players = players_enriched[
    players_enriched["position"] == "Goalkeeper"
]

In [ ]:
attack_strength = (
    attack_players
    .groupby("nationality")
    .apply(
        lambda x: top_n_average(x, 3)
    )
)
midfield_strength = (
    midfield_players
    .groupby("nationality")
    .apply(
        lambda x: top_n_average(x, 4)
    )
)
defense_strength = (
    defense_players
    .groupby("nationality")
    .apply(
        lambda x: top_n_average(x, 4)
    )
)
gk_strength = (
    gk_players
    .groupby("nationality")
    .apply(
        lambda x: top_n_average(x, 1)
    )
)

In [ ]:
team_features = pd.concat(
    [
        attack_strength.rename("attack"),
        midfield_strength.rename("midfield"),
        defense_strength.rename("defense"),
        gk_strength.rename("goalkeeper")
    ],
    axis=1
).fillna(0).reset_index()
team_features["overall_strength"] = (
      0.35 * team_features["attack"]
    + 0.30 * team_features["midfield"]
    + 0.25 * team_features["defense"]
    + 0.10 * team_features["goalkeeper"]
)

In [ ]:
team_features["recent_form"] = team_features[
    "nationality"
].apply(lambda team:
    calculate_recent_form(team, matches)
)


In [ ]:
squad_depth = (
    players_enriched
    .groupby("nationality")
    .apply(
        lambda x: x.nlargest(
            min(23, len(x)),
            "player_strength"
        )["player_strength"].mean()
    )
)
squad_depth = squad_depth.rename(
    "squad_depth"
).reset_index()
team_features = team_features.merge(
    squad_depth,
    on="nationality",
    how="left"
)

In [ ]:
superstar_index = (
    players_enriched
    .groupby("nationality")
    ["player_strength"]
    .max()
    .reset_index()
)
superstar_index = superstar_index.rename(
    columns={
        "player_strength":
        "superstar_index"
    }
)
team_features = team_features.merge(
    superstar_index,
    on="nationality",
    how="left"
)

In [ ]:
team_features["final_team_rating"] = (
      0.75 * team_features["overall_strength"]
    + 0.15 * team_features["squad_depth"]
    + 0.10 * team_features["superstar_index"]
)

In [ ]:
team_features = team_features.fillna(0)

In [ ]:
team_features.sort_values(
    "final_team_rating",
    ascending=False
)[
    [
        "nationality",
        "overall_strength",
        "squad_depth",
        "superstar_index",
        "final_team_rating"
    ]
].head(20)

# Match-Level Dataset Creation

In [ ]:
import pandas as pd

matches = pd.read_csv(
    "../datasets/cleaned/cleaned_matches.csv"
)

team_features = pd.read_csv(
    "../datasets/processed/team_features.csv"
)
matches["date"] = pd.to_datetime(
    matches["date"]
)

In [ ]:
team_lookup = (
    team_features
    .set_index("nationality")
    .to_dict("index")
)

In [ ]:
def get_result(row):

    if row["home_score"] > row["away_score"]:
        return 1

    elif row["home_score"] < row["away_score"]:
        return -1

    return 0
matches["result"] = matches.apply(
    get_result,
    axis=1
)

KeyError: 'elo'

ELO RATING CALCULATION


In [ ]:
matches = matches.sort_values(
    "date"
).reset_index(drop=True)

In [ ]:
INITIAL_ELO = 1500
team_elo = {}
elo_records = []


In [ ]:
def expected_score(rating_a, rating_b):

    return 1 / (
        1 + 10 ** (
            (rating_b - rating_a) / 400
        )
    )

In [ ]:
def actual_result(
    home_score,
    away_score
):

    if home_score > away_score:
        return 1, 0

    elif home_score < away_score:
        return 0, 1

    else:
        return 0.5, 0.5

In [ ]:
def get_k_factor(tournament):

    tournament = str(tournament).lower()

    if tournament == "fifa world cup":
        return 60

    elif "qualification" in tournament:
        return 25

    elif "euro" in tournament:
        return 50

    elif "asian cup" in tournament:
        return 45

    elif "african cup" in tournament:
        return 45

    elif "nations league" in tournament:
        return 35

    return 30

In [ ]:
for _, match in matches.iterrows():

    home_team = match["home_team"]
    away_team = match["away_team"]

    home_score = match["home_score"]
    away_score = match["away_score"]

    match_date = match["date"]

    home_rating = team_elo.get(
        home_team,
        INITIAL_ELO
    )

    away_rating = team_elo.get(
        away_team,
        INITIAL_ELO
    )

    elo_records.append({
        "date": match_date,
        "team": home_team,
        "elo": home_rating
    })

    elo_records.append({
        "date": match_date,
        "team": away_team,
        "elo": away_rating
    })
    HOME_ADVANTAGE = 80
    expected_home = expected_score(
        home_rating + HOME_ADVANTAGE,
        away_rating
    )

    expected_away = expected_score(
        away_rating,
        home_rating + HOME_ADVANTAGE
    )

    actual_home, actual_away = actual_result(
        home_score,
        away_score
    )
    goal_diff = abs(
    home_score - away_score
    )
    margin_multiplier = (
    1 + 0.15 * goal_diff
    )
    K = get_k_factor(
    match["tournament"]
)
    new_home = (
    home_rating
    +
    K * (
        actual_home -
        expected_home
    )
)

    new_away = (
     away_rating
    +
    K * (
        actual_away -
        expected_away
    )
    )

    team_elo[home_team] = new_home
    team_elo[away_team] = new_away

In [50]:
training_rows = []
for _, row in matches.iterrows():

    home_team = row["home_team"]
    away_team = row["away_team"]

    if (
        home_team not in team_lookup or
        away_team not in team_lookup
    ):
        continue

    home = team_lookup[home_team]
    away = team_lookup[away_team]

    feature_row = {

        "attack_diff":
            home["attack"] -
            away["attack"],

        "midfield_diff":
            home["midfield"] -
            away["midfield"],

        "defense_diff":
            home["defense"] -
            away["defense"],

        "goalkeeper_diff":
            home["goalkeeper"] -
            away["goalkeeper"],

        "overall_strength_diff":
            home["overall_strength"] -
            away["overall_strength"],

        "squad_depth_diff":
            home["squad_depth"] -
            away["squad_depth"],

        "superstar_diff":
            home["superstar_index"] -
            away["superstar_index"],

        "final_team_rating_diff":
            home["final_team_rating"] -
            away["final_team_rating"],

        "wins_last5_overall_diff":
            home["wins_last5_overall"] -
            away["wins_last5_overall"],

        "goals_scored_last5_diff":
            home["goals_scored_last5_overall"] -
            away["goals_scored_last5_overall"],

        "goals_conceded_last5_diff":
            home["goals_conceded_last5_overall"] -
            away["goals_conceded_last5_overall"],
        "elo_diff":
            home["elo"] -
            away["elo"],

        "elo_normalized_diff":
            home["elo_normalized"] -
            away["elo_normalized"],

        "enhanced_team_rating_diff":
            home["enhanced_team_rating"] -
            away["enhanced_team_rating"],

        "result":
            row["result"]
    }

    training_rows.append(
        feature_row
    )
    

KeyError: 'elo'

In [ ]:
elo_history = pd.DataFrame(
    elo_records
)

In [ ]:
elo_history.head()

In [ ]:
current_elo = pd.DataFrame({
    "team": list(team_elo.keys()),
    "elo": list(team_elo.values())
})
current_elo = current_elo.rename(
    columns={
        "team": "nationality"
    }
)

In [ ]:
team_features = team_features.merge(
    current_elo,
    on="nationality",
    how="left"
)

In [ ]:
team_features["elo_normalized"] = (
    team_features["elo"]
    -
    team_features["elo"].min()
) / (
    team_features["elo"].max()
    -
    team_features["elo"].min()
)

In [ ]:
team_features["enhanced_team_rating"] = (
      0.60 * team_features["final_team_rating"]
    + 0.40 * team_features["elo_normalized"]
)

In [ ]:
team_lookup = (
    team_features
    .set_index("nationality")
    .to_dict("index")
)

training_rows = []
for _, row in matches.iterrows():

    home_team = row["home_team"]
    away_team = row["away_team"]

    if (
        home_team not in team_lookup or
        away_team not in team_lookup
    ):
        continue

    home = team_lookup[home_team]
    away = team_lookup[away_team]

    feature_row = {

        "attack_diff":
            home["attack"] -
            away["attack"],

        "midfield_diff":
            home["midfield"] -
            away["midfield"],

        "defense_diff":
            home["defense"] -
            away["defense"],

        "goalkeeper_diff":
            home["goalkeeper"] -
            away["goalkeeper"],

        "overall_strength_diff":
            home["overall_strength"] -
            away["overall_strength"],

        "squad_depth_diff":
            home["squad_depth"] -
            away["squad_depth"],

        "superstar_diff":
            home["superstar_index"] -
            away["superstar_index"],

        "final_team_rating_diff":
            home["final_team_rating"] -
            away["final_team_rating"],

        "wins_last5_overall_diff":
            home["wins_last5_overall"] -
            away["wins_last5_overall"],

        "goals_scored_last5_diff":
            home["goals_scored_last5_overall"] -
            away["goals_scored_last5_overall"],

        "goals_conceded_last5_diff":
            home["goals_conceded_last5_overall"] -
            away["goals_conceded_last5_overall"],

        "elo_diff":
            home["elo"] -
            away["elo"],

        "elo_normalized_diff":
            home["elo_normalized"] -
            away["elo_normalized"],

        "enhanced_team_rating_diff":
            home["enhanced_team_rating"] -
            away["enhanced_team_rating"],

        "result":
            row["result"]
    }

    training_rows.append(
        feature_row
    )

In [ ]:
team_features[
    [
        "nationality",
        "elo"
    ]
].head()

In [ ]:
team_features[
    [
        "nationality",
        "final_team_rating",
        "elo",
        "elo_normalized",
        "enhanced_team_rating"
    ]
].head()

In [ ]:
matches["tournament"].value_counts().head(20)

In [ ]:
current_elo[
    current_elo["team"].isin([
        "Brazil"
    ])
].sort_values(
    "elo",
    ascending=False
)

In [ ]:
match_dataset = pd.DataFrame(
    training_rows
)

match_dataset.head()

match_dataset["result"].value_counts()

match_dataset.isnull().sum()

In [46]:
elo_history.to_csv(
    "../datasets/processed/elo_history.csv",
    index=False
)

In [47]:
team_features.to_csv(
    "../datasets/processed/team_features.csv",
    index=False
)

In [48]:
match_dataset.to_csv(
    "../datasets/processed/match_dataset.csv",
    index=False
)

In [ ]:
team_features.head()

In [ ]:
team_features.columns.tolist()